# Exploring a run

This notebook **runs no simulation**: it reads the artifacts of a run already
produced by the pipeline.

```bash
python main.py run --config experiments/full_grid.yaml
```

All the experiment logic lives in `src/pipeline/` and is driven from the command
line. This notebook only serves to explore the results interactively — adding
simulation or configuration code to it would recreate the duplication the
pipeline removed.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

import pandas as pd
from IPython.display import Image, display

from src.pipeline.store import RunStore
from src.pipeline import figures

## Choosing the run

`RunStore.latest()` takes the most recent one; pass a path to `RunStore.open()`
to target another.

In [ ]:
for path in RunStore.list_runs('../results_grid'):
    print(path)

In [ ]:
store = RunStore.latest('../results_grid')
params = store.read_params()
manifest = store.read_manifest()

print(store.root)
print(params.describe())
print(f"seed={params.seed} | commit={manifest['git_commit']} | "
      f"cases={manifest['nb_cases_done']}/{manifest['nb_cases_planned']}")

## Summary table

One row per case (scenario x fleet x method).

In [ ]:
summary = pd.read_csv(store.summary_path)

columns = ['scenario', 'nb_cars', 'method', 'exact_satisfaction',
           'needs_satisfaction', 'mean_travel_distance_km',
           'mean_waiting_time_min', 'total_ms_mean',
           'nb_reservations', 'nb_pres', 'nb_no_show',
           'nb_early_canc', 'nb_late_canc', 'invariant_ok']
summary[columns]

## Comparing the methods

The methods of a given (scenario, fleet) pair run on the **same** initial
world: the measured gap is attributable to the method, not to the draw.

In [ ]:
pivot = summary.pivot_table(
    index=['scenario', 'nb_cars'],
    columns='method',
    values=['exact_satisfaction', 'mean_travel_distance_km', 'total_ms_mean'],
)
pivot

## Figures

Already written by the pipeline in `figures/`. To regenerate them after editing
`src/pipeline/figures.py`:

```bash
python main.py report --latest
```

In [ ]:
for path in sorted(store.figures_dir.glob('*.png')):
    print(path.name)
    display(Image(filename=str(path)))

## Detailed tables

One table per case: `stations`, `behaviors`, `acceptances`, `alpha`, `latency`.

In [ ]:
from src.pipeline.params import CaseParams

case = CaseParams(scenario=params.scenarios[-1],
                  nb_cars=params.fleet_sizes[-1],
                  method='bramev')

stations = pd.read_csv(store.table_path(case, 'stations'))
stations[['station_id', 'nb_charg_spot', 'alpha', 'occupancy_rate',
          'nb_reservations', 'nb_pres', 'nb_no_show', 'nb_late_canc',
          'nb_offer_issued', 'nb_offer_expired']].head(15)

In [ ]:
latency = pd.read_csv(store.table_path(case, 'latency'))
latency[['demand_id', 'nb_stations', 'nb_offers', 'confirmed',
         'first_offer_ms', 'last_offer_ms', 'selection_ms',
         'confirmation_ms', 'total_ms']].describe()

## Drawn intent vs observed outcome

Gap between the probabilities of the scenario and what actually happens.

In [ ]:
behaviors = pd.read_csv(store.table_path(case, 'behaviors'))
behaviors.pivot_table(index='label', columns='kind', values='count', fill_value=0)

In [ ]:
for result in store.iter_results():
    for warning in result['behaviors'].get('diagnostics', []):
        print(f"[{result['scenario']}/{result['config']['nb_cars']}/"
              f"{result['mode']}] {warning}\n")